In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install torchvision torch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()

    self.conv1 = nn.Conv1d(1, 32, 5)
    self.conv2 = nn.Conv1d(32, 64, 5)
    self.conv3 = nn.Conv1d(64, 64, 5)

    self.conv1_bnorm = nn.BatchNorm1d(32)
    self.conv2_bnorm = nn.BatchNorm1d(64)
    self.conv3_bnorm = nn.BatchNorm1d(64)

    self.dropout = nn.Dropout(0.3)

    self.adaptivepool = nn.AdaptiveAvgPool1d(1)

    self.fc = nn.Linear(64, 1)

  def forward(self, x):
    x1 = self.conv1(x)
    x1 = self.conv1_bnorm(x1)
    x1 = F.relu(x1)
    x1 = F.max_pool1d(x1, 2)

    x2 = self.conv2(x1)
    x2 = self.conv2_bnorm(x2)
    x2 = F.relu(x2)
    x2 = F.max_pool1d(x2, 2)

    x3 = self.conv3(x2)
    x3 = self.conv3_bnorm(x3)
    x3 = F.relu(x3)
    x3 = F.max_pool1d(x3, 2)

    x3 = self.adaptivepool(x3)
    x3 = x3.squeeze(-1)
    x3 = self.dropout(x3)
    x3 = self.fc(x3)

    output = x3

    return output

In [ ]:
!pip install h5py

In [ ]:
from torch.utils.data import Dataset
import h5py

with h5py.File('/content/drive/MyDrive/task2/data/Subject01.mat', 'r') as f:
    labels = f['Subject01']['Walking']['Metabolics_System']['Labels']
    print(type(labels))
    print(labels.shape)

# To find label names, dereference HDF5 object references
with h5py.File('/content/drive/MyDrive/task2/data/Subject01.mat', 'r') as f:
    labels = f['Subject01']['Walking']['Metabolics_System']['Labels']
    for i, label in enumerate(labels):
        # dereference the HDF5 reference
        ref = label[0]
        name = ''.join(chr(c) for c in f[ref][:].flatten())
        print(i, name)


In [ ]:
import numpy as np

def calculate_ee(time, vo2, vco2, body_mass):
  mask = time >= 180
  vo2_last3min = vo2[mask]
  vco2_last3min = vco2[mask]
  EE_W = 16.58 * vo2_last3min + 4.51 * vco2_last3min
  EE_W_KG = EE_W / body_mass

  return EE_W_KG.mean()

with h5py.File('/content/drive/MyDrive/task2/data/Subject01.mat', 'r') as f:
    time = f['Subject01']['Walking']['Metabolics_System']['Data'][0]
    vo2 = f['Subject01']['Walking']['Metabolics_System']['Data'][2]
    vco2 = f['Subject01']['Walking']['Metabolics_System']['Data'][3]

calculate_ee(time, vo2, vco2, 63.49)


In [ ]:

subjects = ['Subject01', 'Subject02', 'Subject03', 'Subject04', 'Subject05', 'Subject06', 'Subject07', 'Subject08', 'Subject09', 'Subject10']
activities = ['Walking', 'Running', 'Cycling', 'Backwards', 'Incline']
body_masses = [63.49, 63.49, 71.2, 68.03, 68.03, 68.03, 95.24, 65.76, 68.93, 58.05]

body_masses = {
    'Subject01': 63.49,
    'Subject02': 63.49,
    'Subject03': 71.2,
    'Subject04': 68.03,
    'Subject05': 68.03,
    'Subject06': 68.03,
    'Subject07': 95.24,
    'Subject08': 65.76,
    'Subject09': 68.93,
    'Subject10': 58.05
}

labels = {}


for subject in subjects:
  for activity in activities:
    try:
      with h5py.File(f'/content/drive/MyDrive/task2/data/{subject}.mat', 'r') as f:
        data = f[subject][activity]['Metabolics_System']['Data'][:]
        time = data[0]
        activity_codes = data[1]
        vo2 = data[2]
        vco2 = data[3]
        for code in np.unique(activity_codes):
          mask = (activity_codes == code) & (time >= 180)
          labels[(subject, activity, code)] = calculate_ee(time[mask], vo2[mask], vco2[mask], body_masses[subject])
    except Exception as e:
          print(f"Skipping {subject} {activity}: {e}")
          continue


In [ ]:
import pickle
with open('/content/drive/MyDrive/task2/labels.pkl', 'wb') as f:
    pickle.dump(labels, f)
print("Labels saved")

In [ ]:
for subject in subjects:
    try:
        with h5py.File(f'/content/drive/MyDrive/task2/data/{subject}.mat', 'r') as f:
            print(f"{subject} OK")
    except Exception as e:
        print(f"{subject} CORRUPTED: {e}")

In [ ]:
class EEDataset(Dataset):
    def __init__(self, subjects, labels, sensor_group='Metabolics_System',
                 signal_row=8, window_size=128):
        self.windows = []
        self.targets = []

        for subject in subjects:
            for activity in activities:
                try:
                    with h5py.File(f'/content/drive/MyDrive/task2/data/{subject}.mat', 'r') as f:
                        data = f[subject][activity][sensor_group]['Data'][:]
                        activity_codes = data[1]

                        if isinstance(signal_row, list):
                            signal = np.sqrt(np.sum(data[signal_row]**2, axis=0))
                        else:
                            signal = data[signal_row]

                        for code in np.unique(activity_codes):
                            label = labels.get((subject, activity, code), None)
                            if label is None:
                                continue
                            mask = activity_codes == code
                            signal_condition = signal[mask]

                            for i in range(0, len(signal_condition) - window_size, window_size):
                                window = signal_condition[i:i+window_size]
                                if np.isnan(window).any():
                                    continue
                                self.windows.append(torch.tensor(window, dtype=torch.float32).unsqueeze(0))
                                self.targets.append(torch.tensor(label, dtype=torch.float32))
                except:
                    continue

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx], self.targets[idx]

In [ ]:
test_dataset = EEDataset(['Subject01'], labels)
print(f"Windows: {len(test_dataset)}")
print(f"Sample shape: {test_dataset[0][0].shape}")
print(f"Sample label: {test_dataset[0][1]}")

In [ ]:
from torch.utils.data import DataLoader

train_subjects = ['Subject01', 'Subject02', 'Subject03', 'Subject04', 'Subject05',
                  'Subject06', 'Subject07', 'Subject08', 'Subject09']
test_subject = ['Subject10']

train_dataset = EEDataset(train_subjects, labels)
test_dataset = EEDataset(test_subject, labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print(f"Train windows: {len(train_dataset)}")
print(f"Test windows: {len(test_dataset)}")

In [ ]:
import torch.optim as optim

model = Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for windows, targets in train_loader:
        optimizer.zero_grad()
        predictions = model(windows)
        loss = criterion(predictions.squeeze(1), targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {train_loss/len(train_loader):.4f}")

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.metrics import r2_score
import numpy as np

all_subjects = ['Subject01', 'Subject02', 'Subject03', 'Subject04', 'Subject05',
                'Subject06', 'Subject07', 'Subject08', 'Subject09', 'Subject10']

results = {}

for test_subject in all_subjects:
    train_subjects = [s for s in all_subjects if s != test_subject]

    train_dataset = EEDataset(train_subjects, labels)
    test_dataset = EEDataset([test_subject], labels)

    if len(test_dataset) == 0:
        print(f"Skipping {test_subject} - no data")
        continue

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    model = Net()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

    for epoch in range(20):
        model.train()
        for windows, targets in train_loader:
            optimizer.zero_grad()
            predictions = model(windows)
            loss = criterion(predictions.squeeze(1), targets)
            loss.backward()
            optimizer.step()

    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for windows, targets in test_loader:
            predictions = model(windows)
            preds.extend(predictions.squeeze(1).numpy())
            actuals.extend(targets.numpy())

    preds = np.array(preds)
    actuals = np.array(actuals)
    rmse = np.sqrt(np.mean((preds - actuals)**2))
    mae = np.mean(np.abs(preds - actuals))
    r2 = r2_score(actuals, preds)

    results[test_subject] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"{test_subject}: RMSE={rmse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")

print(f"\nMean RMSE: {np.mean([r['RMSE'] for r in results.values()]):.3f}")
print(f"Mean MAE: {np.mean([r['MAE'] for r in results.values()]):.3f}")
print(f"Mean R²: {np.mean([r['R2'] for r in results.values()]):.3f}")

In [ ]:
with h5py.File('/content/drive/MyDrive/task2/data/Subject01.mat', 'r') as f:
    for group in ['APDM_Accel', 'Empatica_Accel', 'Empatica_Physio', 'EMG']:
        labels_data = f['Subject01']['Walking'][group]['Labels']
        print(f"\n{group} shape: {labels_data.shape}")
        for i, label in enumerate(labels_data):
            ref = label[0]
            name = ''.join(chr(c) for c in f[ref][:].flatten())
            print(f"  {i}: {name}")

In [ ]:
with h5py.File('/content/drive/MyDrive/task2/data/Subject01.mat', 'r') as f:
    print(list(f['Subject01']['Walking']['APDM_Accel'].keys()))
    data = f['Subject01']['Walking']['APDM_Accel']['Data'][:]
    print(data.shape)

In [ ]:
with h5py.File('/content/drive/MyDrive/task2/data/Subject01.mat', 'r') as f:
    for group in ['Empatica_Accel', 'Empatica_Physio', 'EMG']:
        data = f['Subject01']['Walking'][group]['Data'][:]
        labels_data = f['Subject01']['Walking'][group]['Labels']
        print(f"\n{group} — shape: {data.shape}")
        for i, label in enumerate(labels_data):
            ref = label[0]
            name = ''.join(chr(c) for c in f[ref][:].flatten())
            print(f"  {i}: {name}")

In [ ]:
signal_configs = {
    'HeartRate':         ('Metabolics_System', 8),
    'BreathFrequency':   ('Metabolics_System', 5),
    'MinuteVentilation': ('Metabolics_System', 6),
    'SpO2':              ('Metabolics_System', 7),
    'WaistAccel':        ('APDM_Accel', [2,3,4]),
    'ChestAccel':        ('APDM_Accel', [11,12,13]),
    'LeftAnkleAccel':    ('APDM_Accel', [20,21,22]),
    'RightAnkleAccel':   ('APDM_Accel', [29,30,31]),
    'LeftWristAccel':    ('Empatica_Accel', [2,3,4]),    # verify indices
    'RightWristAccel':   ('Empatica_Accel', [5,6,7]),    # verify indices
    'LeftEDA':           ('Empatica_Physio', 2),          # verify indices
    'RightEDA':          ('Empatica_Physio', 3),          # verify indices
    'LeftTemp':          ('Empatica_Physio', 4),          # verify indices
    'RightTemp':         ('Empatica_Physio', 5),          # verify indices
    'LeftEMG':           ('EMG', None),                   # fill after Cell 1
    'RightEMG':          ('EMG', None),                   # fill after Cell 1
}

In [ ]:
def run_loso(sensor_group, signal_row, signal_name=""):
    all_subjects = ['Subject01', 'Subject02', 'Subject03', 'Subject04', 'Subject05',
                    'Subject06', 'Subject07', 'Subject08', 'Subject09', 'Subject10']
    fold_results = []

    for test_subject in all_subjects:
        train_subjects = [s for s in all_subjects if s != test_subject]

        train_dataset = EEDataset(train_subjects, labels, sensor_group, signal_row)
        test_dataset = EEDataset([test_subject], labels, sensor_group, signal_row)

        if len(test_dataset) < 5:
            continue

        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

        model = Net()
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

        for epoch in range(20):
            model.train()
            for windows, targets in train_loader:
                optimizer.zero_grad()
                predictions = model(windows)
                loss = criterion(predictions.squeeze(1), targets)
                loss.backward()
                optimizer.step()

        model.eval()
        preds, actuals = [], []
        with torch.no_grad():
            for windows, targets in test_loader:
                predictions = model(windows)
                preds.extend(predictions.squeeze(1).numpy())
                actuals.extend(targets.numpy())

        preds = np.array(preds)
        actuals = np.array(actuals)

        # handle NaN
        valid = ~(np.isnan(preds) | np.isnan(actuals))
        preds = preds[valid]
        actuals = actuals[valid]

        if len(preds) < 2:
            continue

        rmse = np.sqrt(np.mean((preds - actuals)**2))
        mae = np.mean(np.abs(preds - actuals))
        r2 = r2_score(actuals, preds)
        fold_results.append({'RMSE': rmse, 'MAE': mae, 'R2': r2})

    if fold_results:
        mean_rmse = np.mean([r['RMSE'] for r in fold_results])
        mean_mae = np.mean([r['MAE'] for r in fold_results])
        mean_r2 = np.mean([r['R2'] for r in fold_results])
        print(f"{signal_name}: RMSE={mean_rmse:.3f}, MAE={mean_mae:.3f}, R²={mean_r2:.3f}")
        return mean_rmse, mean_mae, mean_r2
    return None, None, None

In [ ]:
phase1_results = {}

for signal_name, (group, row) in signal_configs.items():
    if row is None:
        continue
    rmse, mae, r2 = run_loso(group, row, signal_name)
    if rmse is not None:
        phase1_results[signal_name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
        # save after each signal
        with open('/content/drive/MyDrive/task2/phase1_results.pkl', 'wb') as f:
            pickle.dump(phase1_results, f)
        print(f"Saved {signal_name}")

HeartRate: RMSE=2.131, MAE=1.799, R²=0.381
Saved HeartRate
BreathFrequency: RMSE=2.333, MAE=1.975, R²=0.399
Saved BreathFrequency
MinuteVentilation: RMSE=1.208, MAE=0.947, R²=0.829
Saved MinuteVentilation
SpO2: RMSE=3.490, MAE=2.844, R²=-0.353
Saved SpO2


In [ ]:
print(phase1_results)

In [ ]:
# Load saved results first
import datetime


with open('/content/drive/MyDrive/task2/phase1_results.pkl', 'rb') as f:
    phase1_results = pickle.load(f)

print(f"Already have: {list(phase1_results.keys())}")

# Run only remaining signals
for signal_name, (group, row) in signal_configs.items():
    if row is None:
        continue
    if signal_name in phase1_results:
        print(f"Skipping {signal_name} — already done")
        continue
    print(f"Starting {signal_name} at {datetime.datetime.now().strftime('%H:%M:%S')}")
    rmse, mae, r2 = run_loso(group, row, signal_name)
    if rmse is not None:
        phase1_results[signal_name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
        with open('/content/drive/MyDrive/task2/phase1_results.pkl', 'wb') as f:
            pickle.dump(phase1_results, f)
        print(f"Done {signal_name} at {datetime.datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("\n=== PHASE 1 COMPLETE ===")
sorted_results = sorted(phase1_results.items(), key=lambda x: x[1]['RMSE'])
for signal, metrics in sorted_results:
    print(f"{signal}: RMSE={metrics['RMSE']:.3f}, MAE={metrics['MAE']:.3f}, R²={metrics['R2']:.3f}")

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

def plot_signal_ranking(phase1_results):
    sorted_results = sorted(phase1_results.items(), key=lambda x: x[1]['RMSE'])
    signals = [r[0] for r in sorted_results]
    rmses = [r[1]['RMSE'] for r in sorted_results]
    r2s = [r[1]['R2'] for r in sorted_results]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    ax1.barh(signals, rmses, color='steelblue')
    ax1.set_xlabel('RMSE (W/kg)')
    ax1.set_title('Phase 1: Signal Ranking by RMSE (lower is better)')

    colors = ['green' if r > 0 else 'red' for r in r2s]
    ax2.barh(signals, r2s, color=colors)
    ax2.set_xlabel('R²')
    ax2.set_title('Phase 1: Signal Ranking by R²')
    ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/task2/phase1_signal_ranking.png', dpi=150)
    plt.show()
    print("Saved.")

In [ ]:
plot_signal_ranking(phase1_results)

In [ ]:
# Split Subject01-08 for train, Subject09 for val, Subject10 for test
train_dataset = EEDataset(['Subject01','Subject02','Subject03','Subject04',
                           'Subject05','Subject06','Subject07','Subject08'],
                           labels, 'APDM_Accel', [29,30,31])
val_dataset = EEDataset(['Subject09'], labels, 'APDM_Accel', [29,30,31])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Train WITH regularization
model_reg = Net()
optimizer = optim.Adam(model_reg.parameters(), lr=0.001, weight_decay=1e-4)
train_losses_reg, val_losses_reg = [], []

for epoch in range(20):
    model_reg.train()
    train_loss = 0
    for windows, targets in train_loader:
        optimizer.zero_grad()
        predictions = model_reg(windows)
        loss = criterion(predictions.squeeze(1), targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model_reg.eval()
    val_loss = 0
    with torch.no_grad():
        for windows, targets in val_loader:
            predictions = model_reg(windows)
            loss = criterion(predictions.squeeze(1), targets)
            val_loss += loss.item()

    train_losses_reg.append(train_loss / len(train_loader))
    val_losses_reg.append(val_loss / len(val_loader))

# Train WITHOUT regularization
class NetNoReg(nn.Module):
    def __init__(self):
        super(NetNoReg, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, 5)
        self.conv2 = nn.Conv1d(32, 64, 5)
        self.conv3 = nn.Conv1d(64, 64, 5)
        self.adaptivepool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(F.max_pool1d(self.conv1(x), 2))
        x = F.relu(F.max_pool1d(self.conv2(x), 2))
        x = F.relu(F.max_pool1d(self.conv3(x), 2))
        x = self.adaptivepool(x).squeeze(-1)
        return self.fc(x)

model_noreg = NetNoReg()
optimizer_noreg = optim.Adam(model_noreg.parameters(), lr=0.001)
train_losses_noreg, val_losses_noreg = [], []

for epoch in range(50):
    model_noreg.train()
    train_loss = 0
    for windows, targets in train_loader:
        optimizer_noreg.zero_grad()
        predictions = model_noreg(windows)
        loss = criterion(predictions.squeeze(1), targets)
        loss.backward()
        optimizer_noreg.step()
        train_loss += loss.item()

    model_noreg.eval()
    val_loss = 0
    with torch.no_grad():
        for windows, targets in val_loader:
            predictions = model_noreg(windows)
            loss = criterion(predictions.squeeze(1), targets)
            val_loss += loss.item()

    train_losses_noreg.append(train_loss / len(train_loader))
    val_losses_noreg.append(val_loss / len(val_loader))

# Plot both
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses_noreg, label='Train Loss')
ax1.plot(val_losses_noreg, label='Val Loss')
ax1.set_title('Without Regularization')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.legend()

ax2.plot(train_losses_reg, label='Train Loss')
ax2.plot(val_losses_reg, label='Val Loss')
ax2.set_title('With Regularization (Dropout + L2)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.legend()

plt.suptitle('Effect of Regularization on Overfitting')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/task2/learning_curves.png', dpi=150)
plt.show()

In [ ]:
# Scatter plot for best signal - RightAnkleAccel
test_dataset = EEDataset(['Subject10'], labels, 'APDM_Accel', [29,30,31])
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

model_best = Net()
optimizer_best = optim.Adam(model_best.parameters(), lr=0.001, weight_decay=1e-4)
train_dataset_best = EEDataset(['Subject01','Subject02','Subject03','Subject04',
                                'Subject05','Subject06','Subject07','Subject08','Subject09'],
                                labels, 'APDM_Accel', [29,30,31])
train_loader_best = DataLoader(train_dataset_best, batch_size=8, shuffle=True)

for epoch in range(20):
    model_best.train()
    for windows, targets in train_loader_best:
        optimizer_best.zero_grad()
        predictions = model_best(windows)
        loss = criterion(predictions.squeeze(1), targets)
        loss.backward()
        optimizer_best.step()

model_best.eval()
preds, actuals = [], []
with torch.no_grad():
    for windows, targets in test_loader:
        predictions = model_best(windows)
        preds.extend(predictions.squeeze(1).numpy())
        actuals.extend(targets.numpy())

preds = np.array(preds)
actuals = np.array(actuals)

plt.figure(figsize=(6,6))
plt.scatter(actuals, preds, alpha=0.6, color='steelblue')
min_val = min(actuals.min(), preds.min())
max_val = max(actuals.max(), preds.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect prediction')
plt.xlabel('Ground Truth EE (W/kg)')
plt.ylabel('Predicted EE (W/kg)')
plt.title('Predicted vs Ground Truth — RightAnkleAccel')
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/task2/scatter_best.png', dpi=150)
plt.show()

In [ ]:
G1 = {
    'WaistAccel':     ('APDM_Accel', [2,3,4]),
    'ChestAccel':     ('APDM_Accel', [11,12,13]),
    'LeftAnkleAccel': ('APDM_Accel', [20,21,22]),
    'RightAnkleAccel':('APDM_Accel', [29,30,31]),
    'LeftWristAccel': ('Empatica_Accel', [2,3,4]),
    'RightWristAccel':('Empatica_Accel', [5,6,7]),
    'LeftEMG':        ('EMG', list(range(2,10))),
    'RightEMG':       ('EMG', list(range(10,18))),
}

G2 = {
    'LeftEDA':           ('Empatica_Physio', 2),
    'RightEDA':          ('Empatica_Physio', 4),
    'LeftTemp':          ('Empatica_Physio', 3),
    'RightTemp':         ('Empatica_Physio', 5),
    'BreathFrequency':   ('Metabolics_System', 5),
    'MinuteVentilation': ('Metabolics_System', 6),
    'HeartRate':         ('Metabolics_System', 8),
    'SpO2':              ('Metabolics_System', 7),
}

G3 = {
    'WaistAccel':        ('APDM_Accel', [2,3,4]),
    'BreathFrequency':   ('Metabolics_System', 5),
    'MinuteVentilation': ('Metabolics_System', 6),
    'HeartRate':         ('Metabolics_System', 8),
}

G4 = {
    'BreathFrequency':   ('Metabolics_System', 5),
    'MinuteVentilation': ('Metabolics_System', 6),
    'HeartRate':         ('Metabolics_System', 8),
    'SpO2':              ('Metabolics_System', 7),
}

G5 = {
    'WaistAccel':        ('APDM_Accel', [2,3,4]),
    'ChestAccel':        ('APDM_Accel', [11,12,13]),
    'LeftAnkleAccel':    ('APDM_Accel', [20,21,22]),
    'RightAnkleAccel':   ('APDM_Accel', [29,30,31]),
    'LeftWristAccel':    ('Empatica_Accel', [2,3,4]),
    'RightWristAccel':   ('Empatica_Accel', [5,6,7]),
}

G6 = {**G1, **G2}  # all 16 signals

In [ ]:
import torch.nn as nn

In [ ]:

class EarlyFusionNet(nn.Module):
    def __init__(self, n_signals):
        super(EarlyFusionNet, self).__init__()
        self.conv1 = nn.Conv1d(n_signals, 32, 5)
        self.conv2 = nn.Conv1d(32, 64, 5)
        self.conv3 = nn.Conv1d(64, 64, 5)
        self.conv1_bnorm = nn.BatchNorm1d(32)
        self.conv2_bnorm = nn.BatchNorm1d(64)
        self.conv3_bnorm = nn.BatchNorm1d(64)
        self.dropout = nn.Dropout(0.3)
        self.adaptivepool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = self.conv1_bnorm(F.relu(F.max_pool1d(self.conv1(x), 2)))
        x = self.conv2_bnorm(F.relu(F.max_pool1d(self.conv2(x), 2)))
        x = self.conv3_bnorm(F.relu(F.max_pool1d(self.conv3(x), 2)))
        x = self.adaptivepool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)

In [ ]:
class IntermediateFusionNet(nn.Module):
    def __init__(self, n_signals):
        super(IntermediateFusionNet, self).__init__()
        self.encoders = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(1, 32, 5),
                nn.BatchNorm1d(32),
                nn.ReLU(),
                nn.MaxPool1d(2),
                nn.Conv1d(32, 64, 5),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1)
            ) for _ in range(n_signals)
        ])
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64 * n_signals, 1)

    def forward(self, x):
        # x shape: (batch, n_signals, time)
        features = [enc(x[:, i:i+1, :]).squeeze(-1) for i, enc in enumerate(self.encoders)]
        combined = torch.cat(features, dim=1)
        combined = self.dropout(combined)
        return self.fc(combined)

In [ ]:
class LateFusionNet(nn.Module):
    def __init__(self, n_signals):
        super(LateFusionNet, self).__init__()
        self.branches = nn.ModuleList([Net() for _ in range(n_signals)])
        self.combine = nn.Linear(n_signals, 1)

    def forward(self, x):
        # x shape: (batch, n_signals, time)
        preds = [branch(x[:, i:i+1, :]) for i, branch in enumerate(self.branches)]
        preds = torch.cat(preds, dim=1)
        return self.combine(preds)

In [ ]:
class MultiSignalDataset(Dataset):
    def __init__(self, subjects, labels, signal_list, window_size=128):
        self.windows = []
        self.targets = []

        for subject in subjects:
            for activity in activities:
                try:
                    signals_data = {}
                    min_len = None
                    activity_codes = None

                    with h5py.File(f'/content/drive/MyDrive/task2/data/{subject}.mat', 'r') as f:
                        for sig_name, (group, row) in signal_list.items():
                            data = f[subject][activity][group]['Data'][:]
                            if activity_codes is None:
                                activity_codes = data[1]
                            if isinstance(row, list):
                                sig = np.sqrt(np.sum(data[row]**2, axis=0))
                            else:
                                sig = data[row]
                            signals_data[sig_name] = sig

                        # resample all signals to same length
                        lengths = [len(s) for s in signals_data.values()]
                        min_len = min(lengths)

                        for sig_name in signals_data:
                            signals_data[sig_name] = signals_data[sig_name][:min_len]
                        activity_codes = activity_codes[:min_len]

                        for code in np.unique(activity_codes):
                            label = labels.get((subject, activity, code), None)
                            if label is None:
                                continue
                            mask = activity_codes == code

                            signal_stack = np.stack([signals_data[s][mask] for s in signals_data])

                            for i in range(0, signal_stack.shape[1] - window_size, window_size):
                                window = signal_stack[:, i:i+window_size]
                                self.windows.append(torch.tensor(window, dtype=torch.float32))
                                self.targets.append(torch.tensor(label, dtype=torch.float32))
                except:
                    continue

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx], self.targets[idx]

In [ ]:
G1 = {
    'WaistAccel':     ('APDM_Accel', [2,3,4]),
    'ChestAccel':     ('APDM_Accel', [11,12,13]),
    'LeftAnkleAccel': ('APDM_Accel', [20,21,22]),
    'RightAnkleAccel':('APDM_Accel', [29,30,31]),
    'LeftWristAccel': ('Empatica_Accel', [2,3,4]),
    'RightWristAccel':('Empatica_Accel', [5,6,7]),
    # EMG excluded due to 1kHz sampling rate computational constraints
}

G2 = {
    'LeftEDA':           ('Empatica_Physio', 2),
    'RightEDA':          ('Empatica_Physio', 4),
    'LeftTemp':          ('Empatica_Physio', 3),
    'RightTemp':         ('Empatica_Physio', 5),
    'BreathFrequency':   ('Metabolics_System', 5),
    'MinuteVentilation': ('Metabolics_System', 6),
    'HeartRate':         ('Metabolics_System', 8),
    'SpO2':              ('Metabolics_System', 7),
}

G3 = {
    'WaistAccel':        ('APDM_Accel', [2,3,4]),
    'BreathFrequency':   ('Metabolics_System', 5),
    'MinuteVentilation': ('Metabolics_System', 6),
    'HeartRate':         ('Metabolics_System', 8),
}

G4 = {
    'BreathFrequency':   ('Metabolics_System', 5),
    'MinuteVentilation': ('Metabolics_System', 6),
    'HeartRate':         ('Metabolics_System', 8),
    'SpO2':              ('Metabolics_System', 7),
}

G5 = {
    'WaistAccel':        ('APDM_Accel', [2,3,4]),
    'ChestAccel':        ('APDM_Accel', [11,12,13]),
    'LeftAnkleAccel':    ('APDM_Accel', [20,21,22]),
    'RightAnkleAccel':   ('APDM_Accel', [29,30,31]),
    'LeftWristAccel':    ('Empatica_Accel', [2,3,4]),
    'RightWristAccel':   ('Empatica_Accel', [5,6,7]),
}

G6 = {**G1, **G2}  # all 16 signals

In [ ]:
def run_fusion_loso(signal_group, fusion_type='early'):
    all_subjects = ['Subject01', 'Subject02', 'Subject03', 'Subject04', 'Subject05',
                    'Subject06', 'Subject07', 'Subject08', 'Subject09', 'Subject10']
    n_signals = len(signal_group)
    fold_results = []

    for test_subject in all_subjects:
        train_subjects = [s for s in all_subjects if s != test_subject]

        train_dataset = MultiSignalDataset(train_subjects, labels, signal_group)
        test_dataset = MultiSignalDataset([test_subject], labels, signal_group)

        if len(test_dataset) < 5:
            continue

        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

        if fusion_type == 'early':
            model = EarlyFusionNet(n_signals)
        elif fusion_type == 'late':
            model = LateFusionNet(n_signals)
        else:
            model = IntermediateFusionNet(n_signals)

        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

        for epoch in range(20):
            model.train()
            for windows, targets in train_loader:
                optimizer.zero_grad()
                predictions = model(windows)
                loss = criterion(predictions.squeeze(1), targets)
                loss.backward()
                optimizer.step()

        model.eval()
        preds, actuals = [], []
        with torch.no_grad():
            for windows, targets in test_loader:
                predictions = model(windows)
                preds.extend(predictions.squeeze(1).numpy())
                actuals.extend(targets.numpy())

        # NaN handling — AFTER the loop, not inside it
        preds = np.array(preds)
        actuals = np.array(actuals)

        valid = ~(np.isnan(preds) | np.isnan(actuals))
        preds = preds[valid]
        actuals = actuals[valid]

        if len(preds) < 2:
            continue

        rmse = np.sqrt(np.mean((preds - actuals)**2))
        mae = np.mean(np.abs(preds - actuals))
        r2 = r2_score(actuals, preds)
        fold_results.append({'RMSE': rmse, 'MAE': mae, 'R2': r2})

    if fold_results:
        return (np.mean([r['RMSE'] for r in fold_results]),
                np.mean([r['MAE'] for r in fold_results]),
                np.mean([r['R2'] for r in fold_results]))
    return None, None, None

In [ ]:

import torch.optim as optim
from sklearn.metrics import r2_score
criterion = nn.MSELoss()


In [ ]:
try:
    with open('/content/drive/MyDrive/task2/phase2_results.pkl', 'rb') as f:
        phase2_results = pickle.load(f)
    print(f"Loaded existing: {list(phase2_results.keys())}")
except:
    phase2_results = {}

groups = {'G1': G1, 'G2': G2, 'G3': G3, 'G4': G4, 'G5': G5, 'G6': G6}

for group_name, signal_group in groups.items():
    if group_name not in phase2_results:
        phase2_results[group_name] = {}
    for fusion_type in ['early', 'intermediate', 'late']:
        if fusion_type in phase2_results.get(group_name, {}):
            print(f"Skipping {group_name} {fusion_type} — already done")
            continue
        print(f"Running {group_name} {fusion_type}...")
        rmse, mae, r2 = run_fusion_loso(signal_group, fusion_type)
        if rmse is not None:
            phase2_results[group_name][fusion_type] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
            print(f"  {group_name} {fusion_type}: RMSE={rmse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")
        with open('/content/drive/MyDrive/task2/phase2_results.pkl', 'wb') as f:
            pickle.dump(phase2_results, f)

In [ ]:
print(len(G1), len(G2), len(G3), len(G4), len(G5), len(G6))
print(EarlyFusionNet, LateFusionNet, IntermediateFusionNet)
print(MultiSignalDataset)
print(len(labels))

In [ ]:
phase2_results = {}
groups = {'G1': G1, 'G2': G2, 'G3': G3, 'G4': G4, 'G5': G5, 'G6': G6}

for group_name, signal_group in groups.items():
    phase2_results[group_name] = {}
    for fusion_type in ['early', 'intermediate', 'late']:
        print(f"Running {group_name} {fusion_type}...")
        rmse, mae, r2 = run_fusion_loso(signal_group, fusion_type)
        if rmse is not None:
            phase2_results[group_name][fusion_type] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
            print(f"  {group_name} {fusion_type}: RMSE={rmse:.3f}, MAE={mae:.3f}, R²={r2:.3f}")
        with open('/content/drive/MyDrive/task2/phase2_results.pkl', 'wb') as f:
            pickle.dump(phase2_results, f)